In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

# Descargar todas las variables
print("Descargando datos...")
oro  = yf.download("GC=F",  start="2020-01-01", end="2024-12-31")
dxy  = yf.download("DX-Y.NYB", start="2020-01-01", end="2024-12-31")
vix  = yf.download("^VIX", start="2020-01-01", end="2024-12-31")
sp500 = yf.download("^GSPC", start="2020-01-01", end="2024-12-31")

# Construir DataFrame combinado
df2 = pd.DataFrame()
df2['oro_retorno']   = oro['Close'].pct_change() * 100
df2['dxy_retorno']   = dxy['Close'].pct_change() * 100
df2['vix_cambio']    = vix['Close'].pct_change() * 100
df2['sp500_retorno'] = sp500['Close'].pct_change() * 100
df2['volatilidad_7d'] = df2['oro_retorno'].rolling(7).std()
df2['retorno_ayer']   = df2['oro_retorno'].shift(1)
df2['ma7']  = oro['Close'].rolling(7).mean()
df2['ma21'] = oro['Close'].rolling(21).mean()

# Limpiar
df2 = df2.dropna()

print(f"\nDataset combinado: {df2.shape[0]} filas, {df2.shape[1]} columnas")
print(df2.head())

Descargando datos...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Dataset combinado: 1237 filas, 8 columnas
            oro_retorno  dxy_retorno  vix_cambio  sp500_retorno  \
Date                                                              
2020-01-31    -0.037889    -0.490450   21.626859      -1.770582   
2020-02-03    -0.360103     0.420992   -4.617839       0.725461   
2020-02-04    -1.699209     0.163595  -10.684475       1.498041   
2020-02-05     0.477298     0.316453   -5.607475       1.125060   
2020-02-06     0.468605     0.234052   -1.254123       0.332567   

            volatilidad_7d  retorno_ayer          ma7         ma21  
Date                                                                
2020-01-31        0.449526      0.872720  1573.985718  1559.933338  
2020-02-03        0.472088     -0.037889  1575.785714  1562.442859  
2020-02-04        0.806435     -0.360103  1572.828578  1562.500006  
2020-02-05        0.820471     -1.699209  1570.114293  1562.100010  
2020-02-06        0.838717      0.477298  1569.528582  1561.780959  


In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Features expandidas
features2 = ['retorno_ayer', 'ma7', 'ma21', 'volatilidad_7d',
             'dxy_retorno', 'vix_cambio', 'sp500_retorno']

X2 = df2[features2]
y2 = df2['oro_retorno']

# Split temporal
split2 = int(len(df2) * 0.8)
X2_train = X2.iloc[:split2]
X2_test  = X2.iloc[split2:]
y2_train = y2.iloc[:split2]
y2_test  = y2.iloc[split2:]

# Entrenar
rf2 = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
rf2.fit(X2_train, y2_train)

# Evaluar
y2_pred_train = rf2.predict(X2_train)
y2_pred_test  = rf2.predict(X2_test)

rmse2_train = np.sqrt(mean_squared_error(y2_train, y2_pred_train))
rmse2_test  = np.sqrt(mean_squared_error(y2_test,  y2_pred_test))
r2_2_train  = r2_score(y2_train, y2_pred_train)
r2_2_test   = r2_score(y2_test,  y2_pred_test)

print("=== MODELO MEJORADO vs MODELO BASE ===")
print(f"\n{'':15} {'Mejorado Train':>15} {'Mejorado Test':>14}")
print("-" * 70)
print(f"{'RMSE':15} {rmse2_train:>14.4f}% {rmse2_test:>13.4f}%")
print(f"{'R²':15} {r2_2_train:>15.4f} {r2_2_test:>14.4f}")
# Importancia de features
print("\n=== IMPORTANCIA DE FEATURES (Modelo Mejorado) ===")
importancias = sorted(zip(features2, rf2.feature_importances_),
                     key=lambda x: x[1], reverse=True)
for feature, imp in importancias:
    barra = '█' * int(imp * 50)
    print(f"{feature:20s}: {imp:.4f} {barra}")

=== MODELO MEJORADO vs MODELO BASE ===

                 Mejorado Train  Mejorado Test
----------------------------------------------------------------------
RMSE                    0.8164%        0.9054%
R²                       0.3948         0.0768

=== IMPORTANCIA DE FEATURES (Modelo Mejorado) ===
dxy_retorno         : 0.4423 ██████████████████████
volatilidad_7d      : 0.2439 ████████████
sp500_retorno       : 0.0883 ████
ma7                 : 0.0860 ████
retorno_ayer        : 0.0613 ███
vix_cambio          : 0.0402 ██
ma21                : 0.0380 █
